# 05 · Flood disruption and network resilience

**Question:** Which routes and residents lose access when low streets close?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Give Astra before/after maps and ask it to distinguish longer travel from complete disconnection.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Threshold inundation is a screening model
This static elevation threshold ignores rainfall, drainage, flow and hydraulic connectivity.
An edge closes when either endpoint is below the threshold. That is conservative but can miss low points between endpoints.


In [ ]:
c=city(); pop=allocate(c['capacity'],1200); nodes,graph=network()
x,y=nodes.T
z=1000+.003*x+.002*y-4*np.exp(-((x-1100)/250)**2)
water_level=1001.0; closed=z<water_level
wet_graph={u:[(v,w) for v,w in edges if not(closed[u] or closed[v])] for u,edges in graph.items()}
idx,connector=snap(c['xy'],nodes); clinic=120
assert not closed[clinic], 'Choose a facility above the water threshold.'
base=(shortest(graph,[clinic])[idx]+connector)/80
flood=(shortest(wet_graph,[clinic])[idx]+connector)/80
reachable=np.isfinite(flood)
assert (flood[reachable]>=base[reachable]-1e-9).all()
unserved=int(pop[~reachable].sum())
print('Residents unreachable:',unserved,'of',pop.sum())
if reachable.any(): print('Mean travel among reachable residents:',np.average(flood[reachable],weights=pop[reachable]))


In [ ]:
fig,ax=plt.subplots()
for u,edges in graph.items():
    for v,w in edges:
        if u<v: ax.plot(*nodes[[u,v]].T,color='#e45756' if closed[u] or closed[v] else '#b9c9ce',lw=1)
ax.scatter(*c['xy'][reachable].T,c='#087f8c',s=15,label='Reachable')
ax.scatter(*c['xy'][~reachable].T,c='#e45756',marker='x',label='Disconnected')
ax.scatter(*nodes[clinic],marker='*',s=160,c='gold',label='Clinic')
map_axes(ax,'Static flood screening'); ax.legend(); plt.show()
export_json('05_disruption.json',dict(synthetic=True,water_level=water_level,unreachable_population=unserved,total_population=int(pop.sum())))


## Teaching lab: Audit disconnection
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 05_disruption.json and the flood-screening map. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Report the fraction of residents disconnected and explain why a finite mean among reachable residents is not a whole-population mean.

**Copyable Codex task:**
> Add a sweep over water thresholds while retaining a dry facility. Plot unreachable population separately from reachable-only travel time. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Removing more edges cannot restore reachability; unreachable residents remain in the total population denominator.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extend with Codex or Astra
Compare several flood levels, alternative facilities and edge-midpoint elevations. Real flood analysis requires validated terrain and hydraulic evidence before planning use.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
